# Deepfake Detection - Train All Strategies (c23)
Make sure to:
- Set **Accelerator → P100** in notebook settings
- Add **both** datasets via **+ Add Input**:
  - `katroue/ff-c23-frames-1` (original, Deepfakes, Face2Face)
  - `katroue/ff-c23-frames-2` (FaceSwap, NeuralTextures, splits)

**To resume Strategy 4 from Phase 2 (skipping Phase 1):**
- Upload `best_model.pth` from your previous session as a Kaggle dataset
- Add it via **+ Add Input** and set `PHASE1_BACKBONE` in Step 5d to its path

In [1]:
# Step 1: Clone repo (shallow clone = latest commit only, fast)
import os, shutil
os.chdir('/kaggle/working')
if os.path.exists('deepfake_project_comp6341'):
    shutil.rmtree('deepfake_project_comp6341')
!git clone --depth 1 --branch training https://github.com/katroue/deepfake_project_comp6341.git
os.chdir('/kaggle/working/deepfake_project_comp6341')
print('Working dir:', os.getcwd())

Cloning into 'deepfake_project_comp6341'...
remote: Enumerating objects: 74, done.
remote: Counting objects: 100% (74/74), done.
remote: Compressing objects: 100% (66/66), done.
remote: Total 74 (delta 13), reused 40 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (74/74), 48.06 KiB | 4.81 MiB/s, done.
Resolving deltas: 100% (13/13), done.
Working dir: /kaggle/working/deepfake_project_comp6341


In [2]:
# Step 2: Install missing dependencies (torch, torchvision, numpy, sklearn already on Kaggle)
!pip install timm grad-cam -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 65.0 MB/s eta 0:00:00:00:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [3]:
# Step 3: Map both Kaggle datasets to expected directory structure
import os

DATASET1 = '/kaggle/input/datasets/katroue/ff-c23-frames-1'  # original, Deepfakes, Face2Face
DATASET2 = '/kaggle/input/datasets/katroue/ff-c23-frames-2'  # FaceSwap, NeuralTextures, splits
DATA_ROOT = '/kaggle/working/deepfake_project_comp6341/data'

dirs = [
    f'{DATA_ROOT}/original_sequences/youtube/c23',
    f'{DATA_ROOT}/manipulated_sequences/Deepfakes/c23',
    f'{DATA_ROOT}/manipulated_sequences/Face2Face/c23',
    f'{DATA_ROOT}/manipulated_sequences/FaceSwap/c23',
    f'{DATA_ROOT}/manipulated_sequences/NeuralTextures/c23',
]
for d in dirs:
    os.makedirs(d, exist_ok=True)

links = {
    f'{DATA_ROOT}/original_sequences/youtube/c23/images':           f'{DATASET1}/original',
    f'{DATA_ROOT}/manipulated_sequences/Deepfakes/c23/images':      f'{DATASET1}/Deepfakes',
    f'{DATA_ROOT}/manipulated_sequences/Face2Face/c23/images':      f'{DATASET1}/Face2Face',
    f'{DATA_ROOT}/manipulated_sequences/FaceSwap/c23/images':       f'{DATASET2}/FaceSwap',
    f'{DATA_ROOT}/manipulated_sequences/NeuralTextures/c23/images': f'{DATASET2}/NeuralTextures',
    f'{DATA_ROOT}/splits':                                           f'{DATASET2}/splits',
}
for link, target in links.items():
    if os.path.islink(link) and not os.path.exists(link):
        os.unlink(link)  # remove broken symlink
    if not os.path.lexists(link):
        os.symlink(target, link)

# Verify
for link in links:
    count = len(os.listdir(link))
    print(f'{link.split("/data/")[1]}: {count} entries')

original_sequences/youtube/c23/images: 1000 entries
manipulated_sequences/Deepfakes/c23/images: 1000 entries
manipulated_sequences/Face2Face/c23/images: 1000 entries
manipulated_sequences/FaceSwap/c23/images: 1000 entries
manipulated_sequences/NeuralTextures/c23/images: 1000 entries
splits: 3 entries


In [4]:
# Step 4: Update configs to point to Kaggle paths
import glob, re

DATA_ROOT = '/kaggle/working/deepfake_project_comp6341/data/'
for cfg in glob.glob('configs/c23/*.yaml'):
    with open(cfg) as f:
        content = f.read()
    content = re.sub(r'data_root:.*', f'data_root: "{DATA_ROOT}"', content)
    content = re.sub(r'save_dir: "results/', 'save_dir: "/kaggle/working/results/', content)
    content = re.sub(r'log_dir: "results/', 'log_dir: "/kaggle/working/results/', content)
    with open(cfg, 'w') as f:
        f.write(content)
    print(f'Updated {cfg}')

Updated configs/c23/stategy2_augmented.yaml
Updated configs/c23/strategy6_multitask.yaml
Updated configs/c23/strategy5_hard_neg.yaml
Updated configs/c23/strategy3_curriculum.yaml
Updated configs/c23/strategy4_ssl.yaml
Updated configs/c23/strategy1_baseline.yaml


In [5]:
# Step 4b: (Optional) Restore checkpoints from a previous session
# To resume training:
#   1. After your previous session, run Step 6 and commit the notebook to save outputs
#   2. In this new session, go to + Add Input → Your Work → find the previous notebook output
#   3. Set CHECKPOINT_DATASET below to the mounted path (check /kaggle/input/ for the name)
#   4. Run this cell, then run the Step 5 cells normally — they will auto-resume

import os, shutil

CHECKPOINT_DATASET = None  # e.g. '/kaggle/input/your-previous-output-name/results'

if CHECKPOINT_DATASET and os.path.exists(CHECKPOINT_DATASET):
    dst = '/kaggle/working/results'
    shutil.copytree(CHECKPOINT_DATASET, dst, dirs_exist_ok=True)
    print(f'Restored checkpoints from {CHECKPOINT_DATASET}')
    for f in ['strategy1_baseline', 'strategy2_augmented', 'strategy3_curriculum',
              'strategy4_ssl', 'strategy5_hard_neg', 'strategy6_multitask']:
        p = f'/kaggle/working/results/models/c23/{f}/last_model.pth'
        print(f'  {"✅" if os.path.exists(p) else "❌"} {f}')
else:
    print('No checkpoint dataset set — starting training from scratch')

No checkpoint dataset set — starting training from scratch


In [6]:
def save_results():
    import shutil, os
    src = '/kaggle/working/results'
    dst = '/kaggle/output/results'
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f'Saved results to {dst}')
    else:
        print('No results directory found yet')

In [ ]:
# Step 5d: Strategy 4 - Self-Supervised (SimSiam)
import torch, os, yaml, shutil
torch.cuda.empty_cache()

# Set to the path of best_model.pth from your uploaded dataset to skip Phase 1
# e.g. '/kaggle/input/strategy4-phase1-checkpoint/best_model.pth'
# Set to None to run Phase 1 from scratch
PHASE1_BACKBONE = '/kaggle/input/datasets/katroue/phase-1-best-model'

if PHASE1_BACKBONE and os.path.exists(PHASE1_BACKBONE):
    # Copy backbone into expected location
    dst = '/kaggle/working/results/models/c23/strategy4_ssl/phase1_pretrained/best_model.pth'
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copy(PHASE1_BACKBONE, dst)
    print(f'Copied Phase 1 backbone to {dst}')

    # Disable Phase 1 so training jumps straight to Phase 2
    cfg_path = 'configs/c23/strategy4_ssl.yaml'
    with open(cfg_path) as f:
        config = yaml.safe_load(f)
    config['phase_1']['enabled'] = False
    with open(cfg_path, 'w') as f:
        yaml.dump(config, f)
    print('Phase 1 disabled — running Phase 2 only')

    !CONFIG_DIR=configs/c23 python -m src.training.train_ssl
else:
    # Full run or resume Phase 2 checkpoint if one exists
    ckpt = '/kaggle/working/results/models/c23/strategy4_ssl/last_model.pth'
    resume = f'--resume {ckpt}' if os.path.exists(ckpt) else ''
    !CONFIG_DIR=configs/c23 python -m src.training.train_ssl {resume}

save_results()

In [ ]:
save_results()

In [ ]:
# Step 6: Copy results to Kaggle output for download
import shutil, os

results_src = '/kaggle/working/results'
if not os.path.exists(results_src):
    print(f'ERROR: {results_src} does not exist — training may have failed. Check step 5 output.')
else:
    shutil.copytree(results_src, '/kaggle/output/results', dirs_exist_ok=True)
    print('Results saved to /kaggle/output/results')
    !find /kaggle/output/results/models -name 'best_model.pth'